In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/mlp-kaggle-assignment-2-may-2026-term/test.csv")
train_df.head(5)

In [ ]:
## shape of train_data
train_df.shape

### 1. Datatypes of diffr columns

In [ ]:
train_df.dtypes

### Takeaways:

<pre>
    
    Column_name       |    About Column_name
    ------------------|----------------------------------------------------------
    #id               |    integer type and increases consequently with each row
    #customer_id      |    8 digit unique identification code for each customer
    #last_name        |    str type in total 2611 unique surname's 
    #credit score     |    float type range between [350.0, 850.0] (null values)
    #country          |    Only three countries(GER, FR, SPA) (null Values)
    #gender           |    str type only have two values (Male, Female
    #age              |    float type must be changed to integer 
    #tenure           |    int type represent num of years
    #acc_balance      |    float, contain null values must be fixed 
    #prod_count       |    float, contain null values must be fixed
    #has_card         |    Boolean columns (0.0, 1.0)
    #is_active        |    Boolean columns (0.0, 1.0)
    #estimated_salary |    float type 
    #exit status      |    target feature boolean (0, 1)
    
</pre>

### 2. Descriptive Statistics for Numerical Columns 

<p style="font-size: 17px;">Quantitative (or Numerical) columns are: <br> credit_score, age, acc_balance, estimated_salary, tenure, prod_count</p>

In [ ]:
num_cols = ["credit_score", "age", "acc_balance", "estimated_salary", "tenure", "prod_count"]

for col in num_cols:
    print(f"{col}: ")
    minimum = min(train_df[col])
    maximum = max(train_df[col])
    mean = train_df[col].mean()
    median = train_df[col].median()
    print("Min: ", minimum)
    print("Max: ", maximum)
    print("Mean: ", mean)
    print("Median: ", median)
    print("\n")


In [ ]:
train_df["tenure"].value_counts()

In [ ]:
train_df["prod_count"].value_counts()

### Takeaways: 

<pre>
    
    credit_score: Mostly average around 656–659, covering normal low to high credit scores.
    age: Most customers are around 37 years old, with a few older people up to 92.
    acc_balance: More than 50% of people have 0 balance in their account.
    estimated_salary: Salary is spread out pretty evenly from very low to around 2 lakhs.
    tenure: Most people have been with the bank for 1 to 9 years in equal numbers.
    prod_count: Almost everyone (~98%) uses only 1 or 2 products. Very few have 3 or 4.

</pre>

### 3. Identify and handle the missing values

In [ ]:
## columns with missing values 
train_df.isna().sum()

#### **Situation:** 

<p style="font-size: 15px;">4 columns have missing values: 
<b>credit_score,
country,
acc_balance,
prod_count</b> </p>

#### **Action:**
<pre>
    credit_score (Continuous Numerical): Use Median.
    country (Categorical): Use Most Frequent (Mode).
    acc_balance (Zero-contained Continuous): Use Median.
    prod_count (Categorical Count): Use Most Frequent (Mode)
</pre>


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

median_cols = ['credit_score', 'acc_balance']
mode_cols = ['country', 'prod_count']

y = train_df["exit_status"]
train_df = train_df.drop(["exit_status"], axis=1)

preprocessor = ColumnTransformer(
    transformers=[
        ('num_median', SimpleImputer(strategy='median'), median_cols),
        ('cat_mode', SimpleImputer(strategy='most_frequent'), mode_cols),
    ],
    remainder='passthrough',
)


preprocessor.set_output(transform='pandas')

train_df_imputed = preprocessor.fit_transform(train_df)
test_df_imputed = preprocessor.transform(test_df)

train_null_vals = train_df_imputed.isna().sum().sum()
test_null_vals = test_df_imputed.isna().sum().sum()

print(f"NA Values in train_df after imputation: {train_null_vals}")
print(f"NA Values in test_df after imputation: {test_null_vals}")

### 4. Identify and handle duplicates

In [ ]:
num_duplicates = train_df_imputed.duplicated().sum()
print(f"Total duplicate rows: {num_duplicates}")

### Takeaway: No Duplicate data available 

### 5. Identify and handle outliers

In [ ]:
train_df_imputed.head(3)

In [ ]:
# concerned cols are only the continuous numerical ones which can influence model behaviour 
concerned_cols = ["num_median__credit_score", 
                  "num_median__acc_balance", 
                  "remainder__age", 
                  "remainder__estimated_salary"]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

outlier_summary = {}
for i, col in enumerate(concerned_cols):
    Q1 = train_df_imputed[col].quantile(0.25)
    Q3 = train_df_imputed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    filter_condition = (train_df_imputed[col] < lower_bound) | (train_df_imputed[col] > upper_bound)
    outliers = train_df_imputed[filter_condition]
    axes[i].boxplot(train_df_imputed[col], tick_labels=[col])

    outlier_summary[col] = {
        "count": len(outliers),
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
    }
    
plt.tight_layout()
plt.show()

In [ ]:
# Summary of outliers

for key, values in outlier_summary.items():
    print(f"{key}: \n {values}")

### 6. Visualizations

#### Target Distribution (exit_status)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))
sns.countplot(x=y)
plt.title('Target Distribution (Churn vs Stay)')
plt.xlabel('Exit Status (0 = Retained, 1 = Churned)')
plt.ylabel('Customer Count')
plt.show()

<br>
<p style="font-size: 15px;">
Takeaway: The dataset is significantly imbalanced <br>
Summary: 80% of customers stayed (0) and only 20% churned (1), accuracy alone is misleading metric.<br>
Inferrence: Evaluate model using ROC-AUC or PR-AUC (Precision-Recall AUC) instead of standard accuracy. </p>


#### Categorical vs. Churn Rate (country & gender vs exit_status)
It shows which customer segments are dropping off the most

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plot_df = train_df_imputed.copy()
plot_df['exit_status'] = y

# Churn Rate by Country
sns.barplot(
    data=plot_df,
    x='cat_mode__country',
    y='exit_status',
    ax=axes[0],
    errorbar=None,
    palette='magma',
)
axes[0].set_title('Churn Rate by Country')
axes[0].set_ylabel('Churn Rate (Proportion)')

# Churn Rate by Gender
sns.barplot(
    data=plot_df,
    x='remainder__gender',
    y='exit_status',
    ax=axes[1],
    errorbar=None,
    palette='viridis',
)
axes[1].set_title('Churn Rate by Gender')
axes[1].set_ylabel('Churn Rate (Proportion)')

plt.show()

<br>
<p style="font-size: 15px;">
Takeaway: <br>
country: Customers in Germany churn at more than double the rate (~38%) compared to France and Spain (~17%). <br>
gender: Female customers (~28%) churn noticeably more than Male customers (~16%).
</p>


#### Churn vs Age and Churn vs Account_balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Age distribution split by Churn
sns.kdeplot(
    data=plot_df,
    x='remainder__age',
    hue='exit_status',
    common_norm=False,
    fill=True,
    ax=axes[0],
    palette='Set1',
)
axes[0].set_title('Age Distribution by Churn')

# Account Balance distribution split by Churn
sns.kdeplot(
    data=plot_df,
    x='num_median__acc_balance',
    hue='exit_status',
    common_norm=False,
    fill=True,
    ax=axes[1],
    palette='Set1',
)
axes[1].set_title('Account Balance Distribution by Churn')

plt.tight_layout()
plt.show()

<p style="font-size:15px;">Takeaway: <br>
age: Older customers (40–55) are at a much higher risk of churning, whereas younger customers (30–38) are mostly retained.

acc_balance: Customers with higher non-zero balances (100k+) are churning more than customers with zero balances, making high-balance older users a critical risk segment. </p>

### Add 7 more columns to make model training better 

In [100]:
import numpy as np
import pandas as pd


def add_engineered_features(df):
    df = df.copy()

    # 1. Balance to Salary Ratio (Financial Health)
    df['balance_to_salary'] = df['acc_balance'] / (df['estimated_salary'] + 1)

    # 2. Is Zero Balance Flag (Differentiates 0 balance from actual holders)
    df['is_zero_balance'] = (df['acc_balance'] == 0).astype(int)

    # 3. Tenure to Age Ratio (Customer Loyalty Ratio)
    df['tenure_to_age'] = df['tenure'] / (df['age'] + 1e-5)

    # 4. Age Groups (Catching the 40-55 high-risk churn peak we saw in plots)
    df['is_high_risk_age'] = (
        (df['age'] >= 40) & (df['age'] <= 55)
    ).astype(int)

    # 5. Products per Tenure (Product Adoption Speed)
    df['prod_per_tenure'] = df['prod_count'] / (df['tenure'] + 1)

    # 6. Active Member x Product Count Interaction (Engagement Metric)
    df['active_x_prod'] = df['is_active'] * df['prod_count']

    # 7. Credit Score to Age Ratio
    df['credit_per_age'] = df['credit_score'] / (df['age'] + 1e-5)

    return df


# Apply to raw dataframes before preprocessing
train_df_fe = add_engineered_features(train_df_imputed)
test_df_fe = add_engineered_features(test_df_imputed)
train_df_fe

KeyError: 'acc_balance'

### 7. Scale Numerical features and Encode Categorical features

In [97]:
train_df_fe.head()

,num_median__credit_score,num_median__acc_balance,cat_mode__country,cat_mode__prod_count,remainder__id,remainder__customer_id,remainder__last_name,remainder__gender,remainder__age,remainder__tenure,remainder__has_card,remainder__is_active,remainder__estimated_salary
0,559.0,149989.39,France,1.0,0,15788291,Iredale,Male,28.0,1,1.0,1.0,67622.46
1,694.0,114510.35,France,2.0,1,15642816,Hs?,Female,37.0,7,0.0,0.0,182797.86
2,585.0,0.00,France,2.0,2,15632272,T'ien,Female,45.0,8,1.0,1.0,170338.35
3,589.0,166082.18,Spain,1.0,3,15685826,Hightower,Male,25.0,0,1.0,1.0,166476.46
4,701.0,0.00,France,2.0,4,15658032,Hopkins,Male,39.0,2,1.0,0.0,82526.92


In [98]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Categorical columns for One-Hot Encoding
ohe_cols = ['country', 'gender']

# 2. Continuous numerical columns to Standard Scale (original + continuous engineered)
scale_cols = [
    'credit_score',
    'acc_balance',
    'age',
    'estimated_salary',
    'tenure',
    'balance_to_salary',  # continuous
    'tenure_to_age',  # continuous
    'prod_per_tenure',  # continuous
    'credit_per_age',  # continuous
]

# 3. Discrete counts & binary flags to Pass Through as-is
passthrough_cols = [
    'prod_count',
    'has_card',
    'is_active',
    'is_zero_balance',  # New binary flag (0/1)
    'is_high_risk_age',  # New binary flag (0/1)
    'active_x_prod',  # New discrete count
]

# Note: 'id', 'customer_id', and 'last_name' are automatically dropped by remainder='drop'

In [99]:
# Create updated ColumnTransformer
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            'ohe',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            ohe_cols,
        ),
        ('scale', StandardScaler(), scale_cols),
        ('pass', 'passthrough', passthrough_cols),
    ],
    remainder='drop',  # Drops id, customer_id, last_name, etc.
)

# Output as pandas DataFrame
final_preprocessor.set_output(transform='pandas')

# Fit on train_df_fe and transform both datasets
X_train_transform = final_preprocessor.fit_transform(train_df_fe)
X_test_transform = final_preprocessor.transform(test_df_fe)

# Ensure numeric types for XGBoost compatibility
X_train_transform = X_train_transform.astype(float)
X_test_transform = X_test_transform.astype(float)

print("Updated Feature Matrix Shape (Train):", X_train_transform.shape)
print("Updated Feature Matrix Shape (Test):", X_test_transform.shape)

ValueError: A given column is not a column of the dataframe

In [ ]:
X_test_transform.head()

### Takeaway
<ul style="font-size: 15px";>
<li>feature matrix is now fully model-ready with 90,000 clean rows and 13 numerical columns (5 standard-scaled, 5 one-hot encoded, and 3 passed-through cols)</li>
<li>All identifiers and missing values have been eliminated, leaving an optimal format for direct input into tree-based or linear classification models.</li> </ul>

### 8. Model Building 

In [ ]:
import lightgbm as lgb
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X_train_transform = X_train_transform.astype(float)
X_test_transform = X_test_transform.astype(float)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_transform, y, test_size=0.2, random_state=42, stratify=y
)

# Aggressive model hyperparameter configuration
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, C=1.0, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, max_depth=16, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=500, max_depth=18, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=6, random_state=42
    ),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.03, max_depth=8, class_weight="balanced", random_state=42
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=500,
        learning_rate=0.02,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(71000 / 19000),
        random_state=42,
        n_jobs=-1,
        eval_metric="auc",
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=600,
        learning_rate=0.02,
        max_depth=8,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(71000 / 19000),
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ),
}

results = {}
test_preds = {}

for name, model in models.items():
    model.fit(X_tr, y_tr)
    val_preds = model.predict_proba(X_val)[:, 1]
    
    # Calculate ROC-AUC score
    score = roc_auc_score(y_val, val_preds)
    results[name] = score
    
    print(f"Model: {name:<25} | Validation ROC-AUC: {score:.5f}")

leaderboard = pd.DataFrame(
    list(results.items()), columns=["Model", "ROC_AUC"]
).sort_values(by="ROC_AUC", ascending=False)

leaderboard

### Takeaway 
<p style="font-size: 15px;">
Gradient Boosting Algorithms dominate: The top 4 positions are all boosting models (XGBoost, Gradient Boosting, Hist Gradient Boosting, and LightGBM), all scoring very close to each other around ~0.874–0.875 ROC-AUC.

Tree Ensembles (Bagging) are second tier: Random Forest (0.865) and Extra Trees (0.859) are solid, but falling behind boosting.</p>

### 9. Hyperparameter Tuning 

In [ ]:
import lightgbm as lgb
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV

# Fast, effective model
param_distributions = {
    "XGBoost": (
        xgb.XGBClassifier(
            scale_pos_weight=(71000 / 19000),
            random_state=42,
            n_jobs=-1,
            eval_metric="auc",
        ),
        {
            "n_estimators": [300, 500],
            "max_depth": [4, 6],
            "learning_rate": [0.02, 0.04],
            "subsample": [0.8],
        },
    ),
    "Hist Gradient Boosting": (
        HistGradientBoostingClassifier(
            class_weight="balanced", random_state=42
        ),
        {
            "max_iter": [200, 300],
            "max_depth": [6, 8],
            "learning_rate": [0.03, 0.05],
        },
    ),
    "LightGBM": (
        lgb.LGBMClassifier(
            scale_pos_weight=(71000 / 19000),
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
        {
            "n_estimators": [300, 500],
            "num_leaves": [31, 63],
            "learning_rate": [0.02, 0.04],
        },
    ),
}

best_models = {}
tuned_results = {}

for name, (model, params) in param_distributions.items():
    print(f"Tuning {name}...")

    # n_iter=3 tries only 3 fast combinations per model
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=3,
        scoring="roc_auc",
        cv=2,  # 2-fold CV for maximum speed
        random_state=42,
        n_jobs=-1,
    )

    search.fit(X_tr, y_tr)

    best_model = search.best_estimator_
    best_models[name] = best_model

    val_preds = best_model.predict_proba(X_val)[:, 1]
    val_score = roc_auc_score(y_val, val_preds)
    tuned_results[name] = val_score

    print(f"-> {name} Best Val ROC-AUC: {val_score:.5f}\n")

# Display quick results
tuned_leaderboard = pd.DataFrame(
    list(tuned_results.items()), columns=["Model", "Tuned_ROC_AUC"]
).sort_values(by="Tuned_ROC_AUC", ascending=False)

tuned_leaderboard

### Takeaway

<ul>
    <li> Default tree settings are already near-optimal: The base XGBoost model scored 0.87538, and tuning brought it to 0.87559—a tiny +0.00021 gain. </li>
    <li> Model ranking remained identical: XGBoost retained 1st place, followed closely by LightGBM and HistGradientBoosting. </li>
</ul>

### 10. Model Prediction on Test data

In [89]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

# Get probability predictions on validation set from your best model
val_preds_prob = xgb_best.predict_proba(X_val)[:, 1]

best_threshold = 0.5
best_f1 = 0.0

for threshold in np.arange(0.1, 0.9, 0.01):
    val_preds_binary = (val_preds_prob >= threshold).astype(int)
    score = f1_score(y_val, val_preds_binary)

    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print(f"🎯 Optimal Threshold found: {best_threshold:.2f}")
print(f"📈 Best Validation F1 Score: {best_f1:.4f}")

# final prediction based on threshold
final_binary_preds = (final_preds >= best_threshold).astype(int)

# Create new submission file
submission = pd.DataFrame({'id': test_df['id'], 'exit_status': final_binary_preds})
submission.to_csv('submission.csv', index=False)

print("\nsubmission.csv created using optimal threshold!")

🎯 Optimal Threshold found: 0.59
📈 Best Validation F1 Score: 0.6446

submission.csv created using optimal threshold!


In [92]:
# Check first few rows and shape
print("Submission Shape:", submission.shape)
print("\nFirst 5 rows:")
print(submission.head(5))

# Ensure no NaN values exist in submission
print(f"\nMissing values in submission: {submission.isna().sum().sum()}")

Submission Shape: (30000, 2)

First 5 rows:
   id  exit_status
0   0            0
1   1            1
2   2            1
3   3            0
4   4            0

Missing values in submission: 0
